# Indeed Job Scraper — Data Science / Canada
Uses Selenium + Brave browser on Linux.

In [29]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
#from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
#from selenium.common.exceptions import NoSuchWindowException, WebDriverException

brave_path   = '/var/lib/flatpak/exports/bin/com.brave.Browser'
driver_path  = '/usr/local/bin/chromedriver-linux64/chromedriver'

options = Options()
options.binary_location = brave_path
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080')
options.add_argument('user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')

options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)

service = Service(executable_path=driver_path)
driver  = webdriver.Chrome(service=service, options=options)
wait    = WebDriverWait(driver, 15)



def scroll_to_the_bottom_of_the_page():
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(2, 4)) 
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
    return 'fin'

domain = 'Data Science'
country = 'Canada'

search_url = f'https://ca.indeed.com/jobs?q={domain.replace(" ","+")}&l={country}'

print(f"Opening: {search_url}")
driver.get(search_url)
time.sleep(100)


data = []
current_page = 0
max_pages = 10
while current_page < max_pages:
    if not is_window_alive(): break
    
    print(f"Processing Page {current_page + 1}...")
    scroll_to_the_bottom_of_the_page()
    time.sleep(2)

    job_cards = driver.find_elements(By.CSS_SELECTOR, 'div.job_seen_beacon')
    
    for card in job_cards:
        try:
            job_title = card.find_element(By.CSS_SELECTOR, 'h2.jobTitle span').text
        except: job_title = 'N/A'
        
        try:
            company_name = card.find_element(By.CSS_SELECTOR, '[data-testid="company-name"]').text
        except: company_name = 'N/A'
        
        try:
            location = card.find_element(By.CSS_SELECTOR, '[data-testid="text-location"]').text
        except: location = 'N/A'

        data.append([job_title, company_name, location, domain, country])

    print(f'Done Page {current_page + 1}: {len(job_cards)} jobs found.')
    current_page += 1

    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a[data-testid="pagination-page-next"]')))
        next_btn.click()
        time.sleep(random.uniform(4, 7))
    except:
        print('No more pages or blocked.')
        break

if data:
    df = pd.DataFrame(data, columns=['job_title', 'company_name', 'location', 'domain', 'country'])
    df = df.drop_duplicates(subset=['job_title', 'company_name'])
    print(f'\nTotal unique jobs scraped: {len(df)}')
    print(df.head())
    df.to_csv('indeed_jobs_clean.csv', index=False)
else:
    print("No data collected.")

driver.quit()

Opening: https://ca.indeed.com/jobs?q=Data+Science&l=Canada
Processing Page 1...
Done Page 1: 16 jobs found.
Processing Page 2...
Done Page 2: 16 jobs found.
Processing Page 3...
Done Page 3: 16 jobs found.
Processing Page 4...
Done Page 4: 16 jobs found.
Processing Page 5...
Done Page 5: 16 jobs found.
Processing Page 6...
Done Page 6: 16 jobs found.
Processing Page 7...
Done Page 7: 16 jobs found.
Processing Page 8...
Done Page 8: 16 jobs found.
Processing Page 9...
Done Page 9: 16 jobs found.
Processing Page 10...
Done Page 10: 16 jobs found.

Total unique jobs scraped: 151
                                     job_title        company_name  \
0                           Research Scientist    Chelsea Avondale   
1                Machine Learning Ops Engineer          Stratum AI   
2                     Data Analyst (Marketing)                CIRA   
3  ML Engineer – Generative AI & LLMs (Remote)  Ample Insight Inc.   
4                    Machine Learning Engineer        Sanctuary AI

In [30]:
df.shape

(151, 5)